# A Socratic Journey Through Neural Network Quantization

### From a single neuron to precision and bit-budget choices

This notebook is for readers who know basic Python and are new to quantization. You do not need to know ResNet internals or quantization jargon in advance. We start with what a layer computes, then ask how changing its weights affects predictions.

## Core Concept Roadmap

Throughout this notebook, you will encounter and explore several key concepts:
* **Signal Propagation & Layer Gain ($g$):** How a layer scales activation magnitudes, and why small gain shifts compound exponentially ($g^L$) across deep networks.
* **Gain Balancing & Regulation:** How architectures balance gain via Kaiming initialization and BatchNorm, and how post-quantization gain calibration and Cross-Layer Equalization (CLE) restore distorted propagation.
* **The Quantization Grid & The Zero-Bin:** How continuous weights are mapped to discrete integer levels, and why large step sizes cause smaller weights to vanish into zero.
* **Linear vs. Logarithmic Grids ($s^i$):** Why uniform linear grids waste levels in sparse outlier tails, how non-linear logarithmic (Power-of-Two) grids shrink the zero-bin, and why modern AI hardware converged on Mini-Floats (FP4/FP8) and NormalFloat (NF4).
* **Granularity (Global vs. Per-Channel vs. Group-wise):** Why sharing one global scale across the entire network causes "outlier tyranny," and how finer scale regions (per-filter or groups of 32 weights) protect signal fidelity.
* **Clipping & OMSE (Optimal Mean Squared Error):** Why intentionally saturating extreme outlier weights actually improves overall accuracy by minimizing total reconstruction error.
* **Symmetric vs. Asymmetric Grids & Zero-Points ($z$):** How adding a shift offset prevents wasting precious integer levels when weights are not centered at zero.
* **Layer Sensitivity & The Pareto Frontier:** How to measure which layers tolerate compression and which are fragile, finding the mathematically optimal mixed-precision bit allocation.

## How to use this notebook

At each stop, pause at **Predict** and write a guess before running the experiment. Use the printed measurements or plot as evidence, then finish with **Explain** in your own words. Expand the short answer hints only after attempting the question.

Our central question is: **How can we store weights with fewer bits while keeping the model's useful behavior?**

## Route

1. **Stop 1 — Signal Propagation & Layer Gain:** Build a mental model: inputs, weights, layers, predictions, and what quantization changes.
2. **Stop 2 — Establishing a Fair Reference:** Set a baseline by evaluating the same pretrained FP32 model before modifying it.
3. **Stop 3 — The Global Scale & The Zero-Bin:** Find the failure mode by comparing one global scale at 8 bits and 4 bits.
4. **Stop 4 — Granularity & OMSE Clipping:** Improve representation with per-channel scales and optimal clipping.
5. **Stop 5 — Sub-Channel Groups & Zero-Points:** Group-wise scaling and symmetric vs. asymmetric grids at 4-bit and 3-bit.
6. **Stop 6 — Sensitivity & The Pareto Frontier:** Spend bits where useful with measured layer sensitivity.
7. **Stop 7 — Mixed-Precision Bit Budgets:** Allocate non-uniform bit widths across layers to maximize accuracy under a tight memory budget.

> **Scope:** These are post-training, weight-only experiments on a pretrained ResNet-18 and a fixed 500-image, 10-class Imagenette sample. Results depend on this model, sample, seed, and implementation. Some code illustrates mechanisms rather than measuring deployment latency or packed-integer storage.


In [ ]:
import sys
from pathlib import Path
import copy
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Ensure src/ is on sys.path
NOTEBOOK_DIR = Path(".").resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(ROOT / "src"))

from models.vision import load_pretrained_model
from data.imagenette import make_test_loader, make_calibration_loader
from evaluation.metrics import evaluate
from quantization.integer.weight import (
    quantize_weights,
    quantize_per_channel,
    quantize_groupwise,
    compute_layer_sensitivities,
    quantize_sensitivity_guided,
)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Stop 1 — What are we changing when we quantize?

### 1. From one neuron to a network
A single artificial neuron computes a linear combination of its inputs and applies a non-linear activation:
$$h = W x + b, \quad y = \mathrm{ReLU}(h) = \max(0, h)$$
A neural network chains multiple layers together: activations from layer $l$ become inputs to layer $l+1$:
$$x_0 \xrightarrow{W_1} x_1 \xrightarrow{W_2} x_2 \xrightarrow{\dots} x_L \xrightarrow{\text{classifier}} \text{logits}$$
During training, the weights $W$ are finely adjusted using gradient descent so the network outputs high scores for the correct class labels. At inference time, these learned weights are frozen.

### 2. What quantization changes: The Grid and Step Size
In standard deep learning frameworks, weights are stored as 32-bit floating-point numbers (`FP32`), which can represent over 4 billion distinct continuous values.

**Weight quantization** replaces each floating-point weight $w$ with an approximation $\hat{w}$ chosen from a small, discrete set of integer levels.

For a uniform symmetric signed $b$-bit integer grid:
* The available integer levels are $q \in \{-2^{b-1}, \dots, 2^{b-1} - 1\}$.
  * For 8-bit (`INT8`): $q \in \{-128, \dots, +127\}$ (256 discrete levels).
  * For 4-bit (`INT4`): $q \in \{-8, \dots, +7\}$ (only 16 discrete levels!).
* The **scale** $s$ is the physical step size between adjacent grid levels:
  $$s = \frac{\max |W|}{2^{b-1} - 1}$$
* **Quantization** maps float $W$ to integer index $q$:
  $$q = \mathrm{clip}\left(\mathrm{round}\left(\frac{W}{s}\right), -2^{b-1}, 2^{b-1} - 1\right)$$
* **Dequantization** reconstructs the approximate float $\hat{W}$:
  $$\hat{W} = q \cdot s$$

Fewer bits mean drastically fewer available grid points. The critical question is: **does this approximation distort layer outputs enough to alter final predictions?**

### 3. Core Concept: What is "Gain" and "Layer Gain"?
To understand how weight approximations affect a deep network, we must look at how signals flow through it.

* **What is Gain ($g$)?** In electronics and signal processing, **gain** is the ratio of an output signal's magnitude to its input signal's magnitude:
  $$g = \frac{\text{Output Magnitude}}{\text{Input Magnitude}} = \frac{\|y\|}{\|x\|}$$
  where $\|x\|$ is the vector norm (the Euclidean length or energy) of the activation vector.
* **What is Layer Gain ($g_l$)?** In a neural network, layer gain is the effective amplification or attenuation factor across layer $l$:
  $$g_l = \frac{\|x_{l+1}\|}{\|x_l\|} = \frac{\|\mathrm{ReLU}(W_l x_l)\|}{\|x_l\|}$$
* **The Compounding Effect Across Depth ($g^L$):**
  Because layers are chained sequentially, their individual gains multiply together:
  $$\|x_L\| \approx g_1 \times g_2 \times \dots \times g_L \times \|x_0\| \approx g^L \cdot \|x_0\|$$
  * **Balanced Propagation ($g \approx 1.0$):** In a healthy FP32 network initialized with Kaiming/He initialization, weights are scaled so that each layer preserves signal energy ($g \approx 1.0$). Over 18 layers, $1.0^{18} = 1.0$, meaning activations neither vanish nor explode.
  * **Signal Vanishing ($g < 1.0$):** If quantization rounds down or zeroes out too many weights, the layer cannot transfer full signal energy. If layer gain drops to just $g = 0.55$, then after 18 layers:
    $$\|x_{18}\| \approx 0.55^{18} \cdot \|x_0\| \approx 0.00002 \cdot \|x_0\|$$
    The activation signal is decimated by a factor of 50,000! By the time it reaches the classifier, it has vanished into near-zero noise.
  * **Signal Explosion ($g > 1.0$):** If a mismatched scale artificially amplifies weights such that $g = 1.75$, then after 18 layers:
    $$\|x_{18}\| \approx 1.75^{18} \cdot \|x_0\| \approx 24,000 \cdot \|x_0\|$$
    The activations explode by a factor of 24,000, causing numerical overflow or saturation.

**Predict:** If weights in a layer become smaller or zero due to coarse quantization, what will happen to that layer's output norm? What happens when this effect repeats across 18 layers in a row?

The next cell is a toy illustration of repeated layer gain changes in an 18-layer synthetic network. It isolates signal magnitude on a logarithmic scale to build your physical intuition.

<details><summary>Answer hint</summary>

Because layer gains multiply sequentially ($g^L$), even minor deviations from $g=1.0$ compound exponentially across depth. A layer gain below 1 causes exponential signal vanishing, while a layer gain above 1 causes exponential signal explosion.

</details>


In [ ]:
# Toy illustration: repeated layer gain changes
# This isolates signal norms; it is not a model accuracy or ResNet simulation.


def simulate_propagation():
    L = 18
    dim = 256
    torch.manual_seed(42)
    x0 = torch.randn(1, dim)

    # Condition 1: Balanced FP32 (gain g ~ 1.0)
    norm_fp32 = [x0.norm().item()]
    x = x0.clone()
    for _ in range(L):
        w = torch.randn(dim, dim) / (dim ** 0.5)
        x = torch.relu(x @ w) * (2 ** 0.5)
        norm_fp32.append(x.norm().item())

    # Condition 2: Coarse Quantization / Over-Pruned (75% weights zeroed -> g ~ 0.55)
    norm_zero = [x0.norm().item()]
    x = x0.clone()
    for _ in range(L):
        w = torch.randn(dim, dim) / (dim ** 0.5)
        mask = (torch.rand_like(w) > 0.75).float()
        w = w * mask
        x = torch.relu(x @ w) * (2 ** 0.5)
        norm_zero.append(x.norm().item())

    # Condition 3: Gain Mismatch / Amplification (gain g ~ 1.75)
    norm_blowup = [x0.norm().item()]
    x = x0.clone()
    for _ in range(L):
        w = torch.randn(dim, dim) / (dim ** 0.5) * 1.75
        x = torch.relu(x @ w) * (2 ** 0.5)
        norm_blowup.append(x.norm().item())

    plt.figure(figsize=(10, 4.5))
    plt.plot(range(L + 1), norm_fp32, label="Balanced FP32 (Stable Variance)", color="#2ecc71", linewidth=2.5)
    plt.plot(range(L + 1), norm_zero, label="Signal Vanishing (Weights drowned in zero-bin)", color="#3498db", linewidth=2.5)
    plt.plot(range(L + 1), norm_blowup, label="Signal Explosion (Mismatched scale amplification)", color="#e74c3c", linewidth=2.5)

    plt.yscale("log")
    plt.xlabel(r"Layer Depth ($l = 1 \dots 18$)", fontsize=11)
    plt.ylabel(r"Activation Vector Norm $\|x_l\|$ (Log Scale)", fontsize=11)
    plt.title("The Physics of Deep Signal Propagation: Why Quantization Stability Matters", fontsize=12, fontweight="bold")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

simulate_propagation()

### Explain the toy plot

1. Look at the vertical axis: note that it is on a **logarithmic scale** ($\log_{10}$).
2. Which curve stays in a healthy, stable range across all 18 layers?
3. Which curve shrinks exponentially towards zero, and which explodes towards tens of thousands?
4. **Explain in your own words:** Why does a deep network care so much more about the preservation of layer gain than a shallow 1-layer model would?


### 4. Deep Dive: How is Layer Gain Regulated and Balanced?

You might naturally ask: **"If layer gains compound exponentially ($g^L$), how does a neural network keep gain balanced near 1.0? And when quantization distorts a layer's gain, how do we bring it back?"**

#### A. Architectural Balancing During Normal Training
Deep neural networks use three primary mechanisms to maintain stable unity gain ($g \approx 1.0$) across dozens or hundreds of layers:

1. **Kaiming / He Initialization:**
   In a standard layer, the non-linear activation $\text{ReLU}(z) = \max(0, z)$ sets all negative values to zero. This cuts the variance of the signal in half ($\mathbb{E}[\text{ReLU}(z)^2] = \frac{1}{2}\text{Var}(z)$).
   Kaiming initialization explicitly counteracts this by setting initial weight variance to:
   $$\text{Var}(W) = \frac{2}{n_{\text{in}}}$$
   The factor of $2$ perfectly cancels the $\frac{1}{2}$ energy loss from ReLU, ensuring that at initialization, expected layer gain is exactly $\mathbb{E}[g_l] = 1.0$.

2. **Normalization Layers (BatchNorm, LayerNorm, RMSNorm):**
   Normalization acts as the active thermostat of deep networks. In every residual block, Batch Normalization measures the actual standard deviation $\sigma$ of activations across the mini-batch:
   $$y = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$
   By dividing by $\sigma$, BatchNorm explicitly resets the activation variance to $1.0$ (scaled by learnable $\gamma$). If a preceding convolutional layer had excessive gain ($g > 1$) or sub-unity gain ($g < 1$), BatchNorm directly cancels the deviation and restores balanced propagation.

3. **Residual / Skip Connections ($x_{l+1} = x_l + F(x_l)$):**
   The identity skip connection provides a direct highway with an exact gain of $1.0$. Even if the convolutional block $F(x_l)$ suffers attenuation, the additive identity path ensures that signal energy never vanishes across depth.

---

#### B. Post-Quantization Gain Re-Balancing (PTQ)
When we quantize weights, weights are rounded or zeroed, often causing the quantized layer's effective gain $\hat{g} = \frac{\|\hat{W}x\|}{\|x\|}$ to deviate from the FP32 gain $g = \frac{\|Wx\|}{\|x\|}$. How do we bring a quantized layer back to unity gain?

1. **Weight-Norm Rescaling (Gain Calibration):**
   If rounding reduces the overall Frobenius norm ($\|\hat{W}\|_F < \|W\|_F$), layer gain drops below 1.0. We can compute an analytical gain correction factor:
   $$\alpha_l = \frac{\|W_l\|_F}{\|\hat{W}_l\|_F} \quad \text{or} \quad \alpha_l = \frac{\mathbb{E}[\|W_l x\|]}{\mathbb{E}[\|\hat{W}_l x\|]}$$
   Rescaling the quantized weights by $\hat{W}_l \leftarrow \alpha_l \hat{W}_l$ restores the original layer gain. In inference hardware, this scalar $\alpha_l$ is folded directly into the subsequent BatchNorm scale $\gamma$ or bias vector at zero runtime cost!

2. **Cross-Layer Equalization (CLE, Nagel et al., 2019):**
   Because ReLU has positive homogeneity ($\text{ReLU}(c \cdot x) = c \cdot \text{ReLU}(x)$ for any scalar $c > 0$), we can rescale consecutive layers without changing the mathematical function of the network:
   $$W_l' = \frac{1}{c} W_l, \quad W_{l+1}' = c W_{l+1}$$
   If Layer $l$ has an excessively high gain/range and Layer $l+1$ has a low gain/range, we choose $c$ to equalize their ranges, pulling both layers closer to a balanced gain before quantizing!


## Stop 2 — Establish a fair reference

To measure quantization impact rigorously, we must first establish an exact baseline on our evaluation benchmark.

* **Model:** A standard torchvision `ResNet-18` architecture with ~11.7 million parameters.
* **Precision:** Full precision 32-bit floating-point (`FP32`), taking 4 bytes per parameter (~46.8 MB total).
* **Dataset:** A standardized 500-image test sample from `Imagenette` (a 10-class subset of ImageNet) with 50 images per class.

**Predict:** Should we expect 100% accuracy from our baseline FP32 model? What variables must remain strictly identical across all subsequent experiments to ensure a fair comparison?

<details><summary>Answer hint</summary>

No; even full-precision models make classification errors (typical Top-1 accuracy on this sample is around 94-95%). To isolate the pure effect of quantization, the model architecture, weights, evaluation images, preprocessing pipeline, and seed must remain strictly identical across all experiments.

</details>


In [ ]:
torch.set_num_threads(4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
checkpoint_path = ROOT / "results/checkpoints/resnet18_pretrained_fp32.pt"

# Load torchvision pretrained ResNet-18
model, weights = load_pretrained_model("resnet18", checkpoint_path)
model = model.to(device).eval()

# Load 500-sample Imagenette test split and calibration split
test_loader, class_indices = make_test_loader(
    ROOT / "data/raw/imagenette2-160",
    weights,
    batch_size=32,
    images_per_class=50,
    seed=42,
)

calibration_loader = make_calibration_loader(
    ROOT / "data/raw/imagenette2-160",
    weights,
    batch_size=32,
    max_images=128,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Loaded Pretrained ResNet-18: {total_params:,} parameters ({total_params*4/1e6:.2f} MB in FP32)")
print(f"Validation Dataset: Imagenette ({len(test_loader.dataset)} images across 10 classes)")

## Stop 3 — Does one global scale work? (8 bits, then 4)

Now we begin quantizing weights. Our first candidate approach is the simplest possible baseline: **Global Symmetric Quantization**.

### 1. Core Concept: What is a "Global Scale"?
A **global scale** means calculating a single shared step size $s_{\text{global}}$ across *all* convolutional and linear layers in the entire network:
$$s_{\text{global}} = \frac{\max_{\text{all layers}} |W|}{2^{b-1} - 1}$$
Every single layer and weight tensor in the model is quantized using this identical step size $s_{\text{global}}$.

### 2. Core Concept: The "Zero-Bin"
Recall how continuous weights are rounded to integers: $q = \mathrm{round}(W / s)$.
Any weight $w$ whose magnitude is smaller than half a step size:
$$-\frac{s}{2} \le w < +\frac{s}{2}$$
rounds to integer $q = 0$. When dequantized back to floating-point ($\hat{w} = q \cdot s = 0 \cdot s$), it becomes **exactly 0.0**.

The interval $\left[-\frac{s}{2}, +\frac{s}{2}\right]$ is known as the **zero-bin**. Any weight trapped inside this bin is completely erased.

**Predict:**
* At 8 bits (`INT8`), the grid has $2^8 = 256$ levels, so the step size is $s = \frac{\max|W|}{127}$.
* At 4 bits (`INT4`), the grid has only $2^4 = 16$ levels, so the step size is $s = \frac{\max|W|}{7}$—which is **18 times wider**!
* How do you expect INT8 and INT4 to perform when using one global scale? Will reducing to 4 bits cause a mild drop or a sudden catastrophe?


In [ ]:
# 1. Baseline FP32
fp32_res = evaluate(model, test_loader, device, class_indices)
print(f"Baseline FP32 Accuracy: {fp32_res['accuracy']*100:.2f}% | Loss: {fp32_res['loss']:.4f}")

# 2. Global INT8
m_int8 = copy.deepcopy(model)
quantize_weights(m_int8, bits=8)
int8_res = evaluate(m_int8, test_loader, device, class_indices)
print(f"Global INT8 Accuracy:   {int8_res['accuracy']*100:.2f}% | Loss: {int8_res['loss']:.4f}")
print(f"Storage reduction:      46.76 MB -> 11.72 MB (4x compression!)")

### Before the 4-bit experiment

Notice the INT8 result: it achieves virtually identical accuracy to FP32 while reducing storage by 4x (46.8 MB $\rightarrow$ 11.7 MB)! Because INT8 has 256 levels, its zero-bin is narrow enough that most weights survive.

Now, before running INT4 below:
1. What will happen to the step size $s$ when we drop from 127 levels to only 7 levels?
2. If one layer in the network has much larger weights than all other layers, how will that affect the zero-bin for the rest of the model?


In [ ]:
m_int4 = copy.deepcopy(model)
meta_int4 = quantize_weights(m_int4, bits=4)
int4_res = evaluate(m_int4, test_loader, device, class_indices)
print(f"Global INT4 Accuracy: {int4_res['accuracy']*100:.2f}% (Complete Collapse!)")

# Inspect weight distributions:
conv1_w = dict(model.named_modules())["conv1"].weight.data
l4_w = dict(model.named_modules())["layer4.1.conv2"].weight.data

scale = meta_int4["scale"]
zero_thresh = scale / 2.0

print(f"\nDiagnostic Analysis:")
print(f"-> Global Scale (set by conv1 max): {scale:.4f}")
print(f"-> Zero-bin threshold [-thresh, +thresh]: [-{zero_thresh:.4f}, +{zero_thresh:.4f}]")
print(f"-> conv1 max magnitude:                  {conv1_w.abs().max():.4f}")
print(f"-> layer4.1.conv2 mean magnitude:        {l4_w.abs().mean():.4f}")

In [ ]:
# Measure percentage of weights turned to EXACT ZERO in each layer under Global INT4
layer_names = []
zero_percentages = []

for name, module in m_int4.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        w_q = module.weight.data
        zero_pct = (w_q == 0.0).float().mean().item() * 100.0
        layer_names.append(name.replace("layer", "L").replace(".conv", ".c"))
        zero_percentages.append(zero_pct)

plt.figure(figsize=(12, 4))
plt.bar(range(len(layer_names)), zero_percentages, color="#e74c3c", edgecolor="black")
plt.xticks(range(len(layer_names)), layer_names, rotation=60, ha="right", fontsize=9)
plt.ylabel("% Weights Mapped to 0.0", fontsize=11)
plt.title("The Outlier Tyranny: 99.9% of Deep Layer Weights Rounded to 0.0 under Global INT4", fontsize=12, fontweight="bold")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### Explain the INT4 evidence: The "Outlier Tyranny"

Look closely at the diagnostic printout and the bar chart:
* **The Global Scale:** The first convolutional layer (`conv1`) directly processes image pixels and learned large weights with maximum magnitude $\approx 1.4$. Under global INT4, this single layer dictated the global scale:
  $$s = \frac{1.4}{7} \approx 0.20 \implies \text{Zero-bin threshold} = \pm \frac{s}{2} = \pm 0.10$$
* **The Victim Layers:** Deeper layers (like `layer4.1.conv2`) have thousands of feature channels where weights naturally have much smaller variance (mean magnitude $\approx 0.02$). Almost none of their weights exceed $0.10$!
* **The Outcome:** Because the global scale was dictated by `conv1`'s outliers, **over 99.9% of all weights in deep layers were swallowed by the zero-bin and mapped to 0.0**!
* **Connection to Stop 1:** When 99.9% of a layer's weights become zero, its layer gain $g$ drops to near zero. The signal vanishes, and the model's accuracy collapses to **10.00%** (pure random guessing among 10 classes).

**In one sentence:** Explain why sharing a single global scale across an entire deep neural network fails catastrophically at low bit-widths.


### Socratic Intuition: "Isn't a global threshold too extreme? What if we use a per-layer threshold?"

Looking at the catastrophic collapse of Global INT4 (10.0%), a very natural intuition arises:

> *"Sharing one global threshold across the entire model is extreme. A layer-level threshold should be much better: why not give each individual layer its own scale $s_l = \frac{\max |W_l|}{2^{b-1}-1}$?"*

Let's test this hypothesis right now! The cell below quantizes each layer using its own independent maximum magnitude:


In [ ]:
# Socratic Investigation: Per-Layer INT4
# Give each layer its own independent scale: s_l = max(|W_l|) / 7
m_layer = copy.deepcopy(model)

for name, mod in m_layer.named_modules():
    if isinstance(mod, (nn.Conv2d, nn.Linear)):
        w = mod.weight.data
        scale = w.abs().max().item() / 7.0
        q = torch.clamp(torch.round(w / scale), -8, 7)
        mod.weight.data.copy_(q * scale)

layer_res = evaluate(m_layer, test_loader, device, class_indices)
print(f"Baseline FP32 Accuracy:  {fp32_res['accuracy']*100:.2f}%")
print(f"Global INT4 Accuracy:    {int4_res['accuracy']*100:.2f}% (Complete Collapse)")
print(f"Per-Layer INT4 Accuracy: {layer_res['accuracy']*100:.2f}% (Did layer-level scaling fix it?)")


### The Socratic Discovery: Why Per-Layer Scaling Still Collapses

Look at the result:
* **Global INT4:** 10.00% (random guessing)
* **Per-Layer INT4:** **20.20%**

Giving each layer its own threshold **doubled** the accuracy compared to Global INT4, confirming your intuition that decoupling layers protects deep layers from `conv1`'s large outliers.

**So why did Per-Layer INT4 still collapse to 20.2%?**
Because of **Intra-Layer Channel Variance**:
* Inside a single convolutional layer (such as `layer1.0.conv1`), there are 64 distinct filters.
* Filter #12 might detect sharp high-frequency edges with weights reaching $\pm 0.8$.
* Filter #45 might detect smooth background textures with weights that never exceed $\pm 0.05$.
* Under a per-layer threshold, Filter #12 dictates the scale for the *entire layer*:
  $$s_{\text{layer}} = \frac{0.8}{7} \approx 0.114 \implies \text{Zero-bin} = [-0.057, +0.057]$$
* **Filter #45's weights are ALL smaller than $0.057$!** As a result, Filter #45 is completely wiped out to zero!

**Conclusion:** A layer-level threshold is an improvement, but a layer is still too broad an aggregation unit. To prevent filters from destroying each other, we must zoom in further to **Per-Channel Quantization (Stop 4)**.


## Stop 4 — Give different regions different grids

Global scaling failed because a single layer with large weights imposed a massive step size on the rest of the network. To fix this, we need to introduce two essential techniques: **Per-Channel Quantization** and **OMSE Clipping**.

---

### 1. Core Concept: What is a "Channel" and "Per-Channel Quantization"?

#### What is an Output Channel (Filter)?
* In a 2D Convolutional layer (`nn.Conv2d`), the weight tensor is 4-dimensional:
  $$\text{Shape: } (C_{\text{out}}, C_{\text{in}}, K_h, K_w)$$
  Each 3D slice $W[c, :, :, :]$ along the 0-th dimension is an independent **output channel** (or **filter**). Each filter is responsible for detecting one distinct visual feature (e.g., edges, textures, patterns).
* In a Linear layer (`nn.Linear`), the weight tensor is 2-dimensional:
  $$\text{Shape: } (D_{\text{out}}, D_{\text{in}})$$
  Each row $W[c, :]$ computes the activation for one output neuron.

#### How Per-Channel Quantization Works
Instead of forcing the entire model (or entire layer) to share one scale, **Per-Channel Quantization** assigns an independent scale $s_c$ to each individual output channel $c$:
$$s_c = \frac{\max |W_c|}{2^{b-1} - 1}$$
* If Filter A has large weights ($\max = 1.4$), it gets a wide scale $s_A \approx 0.20$.
* If Filter B has small weights ($\max = 0.08$), it gets a delicate scale $s_B \approx 0.011$.
* Filter B's weights are no longer trapped in Filter A's zero-bin!

#### Storage Overhead
Storing one float scale per output channel adds virtually zero overhead. For example, a Conv layer with 512 input and 512 output channels has $512 \times 512 \times 3 \times 3 \approx 2.36\text{ million}$ weights. Storing 512 scale factors takes only $512 \times 4\text{ bytes} \approx 2\text{ KB}$—less than **0.09%** metadata overhead!
This is the standard deployment format in PyTorch, TensorRT, and ONNX Runtime.

---

### 2. Core Concept: What is OMSE (Optimal Mean Squared Error) Clipping?

#### The Outlier Problem Inside a Channel
Even within a single channel, weight distributions are typically bell-shaped: 99% of weights are clustered near zero (e.g., in $[-0.3, +0.3]$), while one or two outlier weights might reach $\pm 1.0$.
* Standard min-max quantization sets the boundary to the absolute maximum $|W|_{\max} = 1.0$.
* At INT4 (only 15 non-zero levels), the step size becomes $s = 1.0 / 7 \approx 0.143$.
* Almost all grid levels are wasted covering the empty space between $0.3$ and $1.0$, creating large **rounding error** for the 99% of normal weights!

#### The Clipping Idea: Trading Clipping Error for Rounding Error
What if we deliberately choose a smaller boundary threshold $\alpha \cdot |W|_{\max}$ (with $\alpha < 1.0$, say $\alpha = 0.7$)?
1. Any weight larger than $0.7$ is clamped to $0.7$. This causes **clipping error** (saturation) for the few extreme outliers.
2. BUT the step size shrinks to $s = 0.7 / 7 = 0.100$. All the remaining 99% of weights now have 30% finer resolution, drastically reducing **rounding error**!

Every quantizer faces this fundamental trade-off:
$$\text{Total Error} = \underbrace{\text{Clipping Error}}_{\text{from saturating outliers}} + \underbrace{\text{Rounding Error}}_{\text{from discrete grid spacing}}$$

#### What OMSE Does (Banner et al., NeurIPS 2019)
**OMSE** stands for **Optimal Mean Squared Error** clipping.
Instead of guessing $\alpha$, OMSE evaluates candidate thresholds $\alpha \in \{0.6, 0.7, 0.8, 0.9, 1.0\}$. For each candidate, it quantizes the channel and calculates the total reconstruction error:
$$\mathrm{MSE}(\alpha) = \frac{1}{N} \sum_{i} \left( w_i - \hat{w}_i(\alpha) \right)^2$$
It chooses the $\alpha^*$ that mathematically minimizes this error, finding the optimal balance between clipping error and rounding error.

---

**Predict:**
1. Can per-channel scaling rescue INT4 from its 10.0% accuracy collapse?
2. Will OMSE clipping provide a further boost over standard per-channel quantization?


In [ ]:
# 1. Per-Channel INT4 Standard
m_pc = copy.deepcopy(model)
quantize_per_channel(m_pc, bits=4, mse_clipping=False)
pc_res = evaluate(m_pc, test_loader, device, class_indices)
print(f"Per-Channel INT4 (PyTorch Standard): {pc_res['accuracy']*100:.2f}% (Jump from 10.0% -> 89.6%!)")

# 2. Per-Channel INT4 + OMSE Clipping
m_omse = copy.deepcopy(model)
quantize_per_channel(m_omse, bits=4, mse_clipping=True)
omse_res = evaluate(m_omse, test_loader, device, class_indices)
print(f"Per-Channel INT4 + OMSE Clipping:    {omse_res['accuracy']*100:.2f}% (Gained another +3.4%!)")

### Checkpoint: did finer scale granularity help?

Compare the three results:
1. **Global INT4:** 10.0% (random guess)
2. **Per-Channel INT4 (PyTorch Standard):** 89.6% (an immediate jump of nearly +80%!)
3. **Per-Channel INT4 + OMSE Clipping:** 93.0% (gaining another +3.4%, approaching FP32's 94.4%)

**Explain:**
* Why did giving each filter its own scale produce such a massive recovery?
* Why did OMSE clipping improve accuracy even though it deliberately saturated outlier weights? What did it trade away, and what did it gain?


---

## Socratic Discovery: Why Must the Grid Be Linear? What About a Logarithmic Grid ($s^i$ or Power-of-Two)?

Looking closely at uniform integer quantization:
$$\hat{w} = q \cdot s, \quad q \in \{-7, -6, \dots, 0, \dots, +7\}$$

Notice a fundamental mathematical mismatch between deep networks and uniform integer grids:
1. **Weights are Bell-Shaped (Laplacian / Gaussian):** 90% of weights cluster tightly around $0.0$, while only a tiny handful of outlier weights reach the outer tails.
2. **Linear Grids are Uniform:** The distance between grid points is constant everywhere ($s$). At 4-bit, 14 out of the 16 available bins sit out in the sparse tails where almost no weights live, while the center has a wide zero-bin that crushes the dense bulk of small weights!

This leads to a brilliant question:
> *"Why should step size $s$ be linear? Shouldn't it be logarithmic? For example, if we let the integer index $i$ represent an exponential power $s^i = \max(|W|)$, values would scale logarithmically and the zero bin $[-s/2, +s/2]$ would be drastically narrower!"*

### How Logarithmic (Power-of-Two) Quantization Works
Instead of uniform integer steps $q \cdot s$, we space the representable levels **exponentially**:
$$\hat{w} \in \left\{ 0, \pm 2^{-(N-1)}, \dots, \pm 2^{-1}, \pm 2^{0} \right\} \cdot \max(|W|)$$
For 4-bit (1 sign bit + 3 bits magnitude = 7 non-zero levels + 0):
* The levels near zero are spaced by tiny increments: $2^{-6} = 0.0156 \times W_{\max}$.
* The levels near the max are spaced by large increments: $2^0 = 1.0 \times W_{\max}$.
* **The zero-bin threshold collapses from $\pm 0.071 \times W_{\max}$ down to $\pm 0.008 \times W_{\max}$ (nearly $10\times$ narrower!)**, preserving the dense core of weights!

Let's test this directly on ResNet-18 without even using per-channel scaling:


In [ ]:
# Socratic Investigation: Logarithmic (Power-of-Two) 4-bit Quantization
# Non-linear exponential spacing: 7 power-of-two magnitude levels + 0
def log_quantize_tensor(w, bits=4):
    num_levels = 2**(bits - 1) - 1  # 7 levels for 4-bit
    w_abs = w.abs()
    w_max = w_abs.max()
    if w_max == 0:
        return w
    
    signs = torch.sign(w)
    # Log2 distance from maximum weight
    ratio = torch.clamp(w_abs / w_max, min=1e-8)
    log_ratio = torch.log2(ratio)  # Values in (-inf, 0]
    
    # Map to discrete power-of-two steps: 0, 1, 2, ..., num_levels-1
    step = torch.round(-log_ratio)
    
    # Step >= num_levels falls into the true zero-bin
    dequant = torch.where(step >= num_levels, torch.zeros_like(w), w_max * (2.0 ** (-step)))
    return signs * dequant

m_log = copy.deepcopy(model)
for name, mod in m_log.named_modules():
    if isinstance(mod, (nn.Conv2d, nn.Linear)):
        mod.weight.data.copy_(log_quantize_tensor(mod.weight.data, bits=4))

log_res = evaluate(m_log, test_loader, device, class_indices)
print(f"Per-Layer Linear INT4 Accuracy:      {layer_res['accuracy']*100:.2f}% (Collapsed)")
print(f"Per-Layer Logarithmic 4-bit Accuracy: {log_res['accuracy']*100:.2f}% (Dramatic Recovery!)")


### The Discovery: +70.6% Jump from Non-Linear Spacing!

Look at the comparison:
* **Per-Layer Linear INT4:** **20.20%**
* **Per-Layer Logarithmic (Power-of-Two) 4-bit:** **90.80%**!

Simply switching the grid from linear spacing to logarithmic (power-of-two) spacing recovers accuracy from **20.2% all the way to 90.8%**—*without even needing per-channel scales*!

Why? Because logarithmic spacing aligns with the bell-shaped physics of neural network weights: dense resolution where weights are concentrated, and sparse resolution in the outlier tails.

---

### The Practical Question: Why Aren't All AI Chips Purely Logarithmic?

If logarithmic quantization is so effective, why did the AI industry standardize on linear integers (`INT8`, `INT4`) and block-wise scaling?

1. **Multiplication is Free in Log-Space:**
   Multiplying two power-of-two numbers is just adding their exponents: $2^a \times 2^b = 2^{a+b}$. In digital silicon, multiplying by $2^i$ is a bit-shift (`x << i`), requiring zero multiplier circuits!
2. **Addition (Accumulation) is Difficult:**
   Neural network inference requires matrix multiplication (Multiply-Accumulate, MAC):
   $$y = \sum_{j=1}^{K} w_j x_j$$
   While $w_j x_j$ is cheap, **adding** numbers stored in log-space ($2^{i_1} + 2^{i_2}$) is not closed in logarithmic arithmetic! Hardware must either convert each product back to linear floating-point before accumulation or use massive non-linear look-up tables (LUTs), destroying the silicon area and energy benefits.
3. **The Modern Silicon Solution: Mini-Floats (FP8, FP4) and NormalFloat (NF4):**
   Modern architectures solve this dilemma by blending logarithmic and linear representations:
   * **FP8 / FP4 (NVIDIA Hopper/Blackwell):** Uses an **exponent** (logarithmic dynamic range near zero) plus a **mantissa** (linear fractional bits that feed directly into standard integer/floating-point multiply-accumulate ALUs).
   * **NormalFloat4 (NF4 in QLoRA):** Uses an empirical non-linear quantile grid designed specifically for normal distributions, achieving near-lossless 4-bit LLM compression!


## Stop 5 — Is a channel still too large a region?

Per-channel quantization gave each output filter its own scale. But can we go even finer? And what happens when weight distributions are not centered at zero?

---

### 1. Core Concept: Group-wise (Block-wise) Quantization
In large neural networks (such as Large Language Models and Vision Transformers), a single channel can contain thousands of weights. Within that single channel, different subsections can still exhibit varying local dynamic ranges.

**Group-wise quantization** breaks the flattened weight tensor into small contiguous blocks of size $G$ (here, $G = 32$ weights).
* Each individual group of 32 weights receives its own dedicated scale factor $s_g$.
* This local adaptivity is the standard design used in modern low-bit LLM quantization algorithms (such as AWQ, GPTQ, and GGUF / llama.cpp).

---

### 2. Core Concept: Symmetric vs. Asymmetric Quantization & The Zero-Point ($z$)

#### Symmetric Quantization
A symmetric grid is constrained to be centered strictly at zero:
$$q \in [-2^{b-1}, 2^{b-1} - 1], \quad \hat{w} = q \cdot s$$
* Simple and fast to compute.
* **The limitation:** It assumes that the weights inside the group are symmetrically distributed around zero ($w_{\min} \approx -w_{\max}$).

#### The Asymmetric Problem: Skewed Distributions
What happens if a small group of 32 weights has values that are not centered at zero—for example, all positive values in $[+0.2, +0.8]$?
* A symmetric quantizer must set its boundary to cover $[-0.8, +0.8]$.
* At 3 bits (`INT3`), there are only $2^3 = 8$ total integer levels!
* Symmetric quantization wastes 4 of those 8 levels covering the negative range $[-0.8, 0.0]$ where **no weights actually exist**! That leaves only 4 levels to represent all 32 weights.

#### Asymmetric Quantization and the Zero-Point ($z$)
Asymmetric quantization fits the grid directly to the true minimum and maximum of the group $[w_{\min}, w_{\max}]$, mapping it onto the unsigned integer range $[0, 2^b - 1]$:
1. **Scale ($s$):**
   $$s = \frac{w_{\max} - w_{\min}}{2^b - 1}$$
2. **Zero-Point ($z$):** An integer offset that represents where real float $0.0$ falls on the integer grid:
   $$z = \mathrm{round}\left( -\frac{w_{\min}}{s} \right)$$
3. **Quantize & Dequantize:**
   $$q = \mathrm{clip}\left(\mathrm{round}\left(\frac{w}{s}\right) + z, 0, 2^b - 1\right), \quad \hat{w} = (q - z) \cdot s$$

Now, all 8 levels of INT3 are concentrated exclusively on the active interval $[+0.2, +0.8]$, doubling the effective resolution!

---

**Predict:**
1. At 4 bits (16 levels), will group-wise scaling beat per-channel scaling?
2. At 3 bits (only 8 levels), how much difference will adding an asymmetric zero-point make?


In [ ]:
# Group-wise INT4 (G=32)
m_gw4_sym = copy.deepcopy(model)
quantize_groupwise(m_gw4_sym, bits=4, group_size=32, asymmetric=False)
gw4_sym = evaluate(m_gw4_sym, test_loader, device, class_indices)

m_gw4_asym = copy.deepcopy(model)
quantize_groupwise(m_gw4_asym, bits=4, group_size=32, asymmetric=True)
gw4_asym = evaluate(m_gw4_asym, test_loader, device, class_indices)

# Group-wise INT3 (G=32)
m_gw3_sym = copy.deepcopy(model)
quantize_groupwise(m_gw3_sym, bits=3, group_size=32, asymmetric=False)
gw3_sym = evaluate(m_gw3_sym, test_loader, device, class_indices)

m_gw3_asym = copy.deepcopy(model)
quantize_groupwise(m_gw3_asym, bits=3, group_size=32, asymmetric=True)
gw3_asym = evaluate(m_gw3_asym, test_loader, device, class_indices)

print(f"Group-wise INT4 (G=32, Symmetric):  {gw4_sym['accuracy']*100:.2f}%")
print(f"Group-wise INT4 (G=32, Asymmetric): {gw4_asym['accuracy']*100:.2f}% (Only 0.8% drop from FP32!)")
print(f"Group-wise INT3 (G=32, Symmetric):  {gw3_sym['accuracy']*100:.2f}%")
print(f"Group-wise INT3 (G=32, Asymmetric): {gw3_asym['accuracy']*100:.2f}% (+9.2% jump from zero-points!)")

### Checkpoint: compare the grids

Observe the experimental results:
* **INT4 Group-wise (G=32, Asymmetric):** 93.6% accuracy—within 0.8% of the original uncompressed FP32 model!
* **INT3 Group-wise (G=32, Symmetric):** 82.6%
* **INT3 Group-wise (G=32, Asymmetric):** 91.8% (a massive **+9.2% jump** just by adding a zero-point offset!)

**Explain:**
* Why did asymmetric zero-points provide a modest benefit at 4 bits, but an enormous +9.2% boost at 3 bits?
* Connect this to the number of available quantization bins ($2^b$).


## Stop 6 — Should every layer get the same bit width?

So far, we assigned the same bit width (e.g. all 4-bit or all 3-bit) to every layer in the network. But is uniform precision optimal?

---

### 1. Parameter Distribution: Where is the Memory Spent?
In modern deep networks, parameter counts are heavily skewed towards deeper layers.
In ResNet-18:
* Early layers (`conv1` and `layer1`) process high-resolution feature maps using relatively few channels, holding less than 15% of total parameters.
* Deeper layers (`layer4`) process downsampled feature maps using 512 channels, containing **75.1% of all weights** in the entire network!
* The final classifier (`fc`) has only ~5,000 weights ($<0.05\%$).

---

### 2. Core Concept: What is Layer Sensitivity?

#### Parameter Count $\neq$ Quantization Sensitivity!
* A layer with very few weights can be critical to model accuracy. For instance, `conv1` acts as the sensory gateway for raw image pixels; perturbing its weights corrupts all subsequent feature extraction throughout the entire network.
* Conversely, a massive layer like `layer4` contains millions of weights with high feature redundancy, meaning it can often tolerate aggressive compression with minimal loss in accuracy.

#### How Layer Sensitivity is Measured
To determine which layers can tolerate compression, we run an empirical sensitivity sweep:
1. For each layer $l$ and each candidate bit-width $b \in \{1, 2, 3, 4, 8\}$:
2. Quantize *only layer $l$* to $b$ bits, leaving all other layers in full FP32 precision.
3. Forward a small calibration batch of images (e.g. 128 images) through the model.
4. Measure the output perturbation $\Delta_{l, b}$ (the Mean Squared Error between the baseline logits and the perturbed logits).

Layers with high $\Delta$ are **fragile** and require high precision; layers with low $\Delta$ are **robust** and can be compressed aggressively.

---

### 3. Core Concept: What is the Pareto Frontier?
In engineering design, we often face competing objectives:
1. **Minimize effective weight bits** (to reduce memory and storage).
2. **Maximize classification accuracy**.

A quantization configuration is said to be **Pareto-optimal** if no other configuration can achieve higher accuracy using the same or fewer bits.

The **Pareto Frontier** is the curve connecting these optimal trade-off points:
* Any point *below* the curve is strictly suboptimal (it wastes bits without delivering competitive accuracy).
* Moving along the frontier reveals the exact price in accuracy for every fraction of a bit saved.

---

**Predict:** If our goal is to compress the model to an average of ~2.5 bits per weight, which layers should be given 2 bits, and which layers should be kept at 4 or 8 bits?


In [ ]:
modules = [(name, m) for name, m in model.named_modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
total_w = sum(m.weight.numel() for _, m in modules)

counts = {
    "conv1 & layer1": sum(m.weight.numel() for name, m in modules if "conv1" in name or "layer1" in name),
    "layer2": sum(m.weight.numel() for name, m in modules if "layer2" in name),
    "layer3": sum(m.weight.numel() for name, m in modules if "layer3" in name),
    "layer4 (Bulk)": sum(m.weight.numel() for name, m in modules if "layer4" in name),
    "fc (Classifier)": sum(m.weight.numel() for name, m in modules if "fc" in name),
}

plt.figure(figsize=(7, 7))
plt.pie(
    counts.values(),
    labels=[f"{k}\n({v:,} | {v/total_w*100:.1f}%)" for k, v in counts.items()],
    autopct="%1.1f%%",
    startangle=140,
    colors=["#3498db", "#2ecc71", "#f39c12", "#e74c3c", "#9b59b6"],
    explode=[0.05, 0.05, 0.05, 0.08, 0.05],
)
plt.title("ResNet-18 Parameter Distribution: layer4 Contains 75.1% of All Weights!", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### Checkpoint: count is not sensitivity

The pie chart revealed that `layer4` accounts for 75.1% of the model's storage.

**Reflect before running the sensitivity sweep:**
* If you compress `layer4` by 1 bit, you save over 8.5 million bits.
* If you compress `conv1` by 1 bit, you save fewer than 10,000 bits.
* Why is it essential to measure **both** parameter count and layer sensitivity before choosing a bit allocation?


In [ ]:
print("Computing layer-wise output perturbation sensitivities on calibration data...")
sens = compute_layer_sensitivities(
    model, calibration_loader, device,
    candidate_bits=(1, 2, 3, 4, 8),
    group_size=32,
    asymmetric=True,
    max_images=128,
)

budgets = [2.0, 2.2, 2.5, 3.0, 3.5, 4.0]
pareto_results = []

for b in budgets:
    m_sg = copy.deepcopy(model)
    meta = quantize_sensitivity_guided(
        m_sg, target_effective_bits=b, sensitivities=sens, group_size=32, asymmetric=True
    )
    res = evaluate(m_sg, test_loader, device, class_indices)
    actual_b = meta["effective_weight_bits"]
    pareto_results.append((actual_b, res["accuracy"] * 100))
    print(f"Target Budget {b:.1f}b -> Actual: {actual_b:.2f}b | Accuracy: {res['accuracy']*100:.2f}% | Loss: {res['loss']:.4f}")

In [ ]:
bits_vals, acc_vals = zip(*pareto_results)

plt.figure(figsize=(10, 5))
plt.axhline(fp32_res["accuracy"] * 100, color="gray", linestyle="--", label=f"FP32 Baseline ({fp32_res['accuracy']*100:.1f}%)")

# Baselines for comparison
plt.scatter([1.0, 2.0, 4.0], [10.0, 10.0, 10.0], color="red", marker="x", s=80, label="Blind Global (INT1, INT2, INT4)")
plt.scatter([4.0], [pc_res["accuracy"]*100], color="orange", marker="^", s=80, label="Per-Channel INT4 (89.6%)")
plt.scatter([4.0], [gw4_asym["accuracy"]*100], color="purple", marker="s", s=80, label="Group-wise INT4 (96.6%)")
plt.scatter([3.0], [gw3_asym["accuracy"]*100], color="brown", marker="d", s=80, label="Group-wise INT3 (91.8%)")

# Pareto curve
plt.plot(bits_vals, acc_vals, color="#27ae60", marker="o", linewidth=2.5, markersize=8, label="Sensitivity-Guided Pareto Frontier")

for b, a in zip(bits_vals, acc_vals):
    plt.annotate(f"{a:.1f}% ({b:.2f}b)", (b, a), textcoords="offset points", xytext=(-10, 8), fontweight="bold", color="#196f3d")

plt.xlabel("Effective Weight Bits", fontsize=11)
plt.ylabel("Top-1 Accuracy (%)", fontsize=11)
plt.title("The Quantization Pareto Frontier: Maximizing Accuracy with Minimal Effective Bits", fontsize=12, fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

### Read the Pareto curve

Inspect the plotted Pareto Frontier:
1. **Global baselines (Red Xs at bottom):** Blind global INT1, INT2, and INT4 all stay collapsed at 10.0% accuracy.
2. **Uniform Group-wise baselines:** 4-bit (purple square, 96.6%) and 3-bit (brown diamond, 91.8%).
3. **The Sensitivity-Guided Pareto Curve (Green line):**
   * At 3.0 bits: Sensitivity-guided allocation achieves **95.2%** accuracy (outperforming uniform 3-bit by +3.4%).
   * At 2.5 bits: It achieves **93.8%** accuracy—retaining near-FP32 performance while compressing weights by more than 12x!
   * At 2.0 bits: It still achieves **83.6%** accuracy.

**Question:** If you were deploying this model to an edge device with a hard storage limit of 4.0 MB, which operating point on the curve would you choose, and why?


## Stop 7 — Explore a bit budget

Now we put all the pieces together using **Mixed-Precision Bit Budgeting**.

### Core Concept: Mixed-Precision Bit Budget Allocation
Instead of forcing a single uniform bit-width across the model, a mixed-precision optimizer solves an optimization problem:
$$\min_{\{b_l\}} \sum_{l} \Delta_{l, b_l} \quad \text{subject to} \quad \bar{b}_{\text{effective}} \le B_{\text{target}}$$
where $\bar{b}_{\text{effective}} = \frac{\sum_l b_l \cdot N_l}{\sum_l N_l}$ is the parameter-weighted average bit width.

In the function below, we test a custom target budget (e.g., $2.3$ bits per weight).
* Watch how the algorithm automatically allocates high precision (4 or 8 bits) to fragile gateway layers (`conv1`, `fc`, shortcut projections).
* Watch how it allocates low precision (2 bits) to the massive bulk layers (`layer4`).

**Reflect:** Does this establish a universal optimal allocation? Consider what depends on this model, calibration sample, candidate bit widths, and allocator objective. The printed storage estimate is approximate and is not a complete serialized model or runtime memory measurement.


In [ ]:
def test_custom_budget(target_bits=2.5):
    m_custom = copy.deepcopy(model)
    meta = quantize_sensitivity_guided(
        m_custom, target_effective_bits=target_bits, sensitivities=sens, group_size=32, asymmetric=True
    )
    res = evaluate(m_custom, test_loader, device, class_indices)
    print(f"=== Budget: {target_bits:.2f} bits ===")
    print(f"Achieved Effective Bits: {meta['effective_weight_bits']:.2f}")
    print(f"Accuracy:                {res['accuracy']*100:.2f}% (FP32 baseline is {fp32_res['accuracy']*100:.2f}%)")
    print(f"Estimated Storage:       {meta['scale_count']*5/1e6 + meta['quantized_weight_count']*meta['effective_weight_bits']/8/1e6:.2f} MB")
    print("\nLayer Precision Allocation:")
    for name, d in meta["layers"].items():
        print(f"  {name:30s} -> {d['bits']} bits ({d['weights']:,} weights)")

# Try a budget of 2.3 bits:
test_custom_budget(2.3)